# 02 Makemore Bigrams

A lecture 2 build log for a character-level bigram language model.

## Scope
- rebuild the count-based bigram baseline from scratch
- evaluate with average negative log-likelihood
- compare the count model against a neural bigram across train-set sizes

## Research Delta
Hold the modeling family fixed at bigrams, and change only the parameterization:

- baseline: add-one smoothed count matrix
- delta: learned neural bigram logits
- controlled variable: amount of training data


In [1]:
%matplotlib inline

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.makemore_bigram import (
    average_negative_log_likelihood,
    build_vocab,
    count_bigram_matrix,
    evaluate_data_regimes,
    load_words,
    sample_from_prob_matrix,
    split_words,
    train_neural_bigram,
)

DATA_PATH = ROOT / "data" / "names.txt"
SEED = 42
torch.manual_seed(SEED)


In [5]:
words = open(DATA_PATH, 'r').read().splitlines()
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [6]:
len(words)

32033

In [7]:
min(len(w) for w in words)

2

In [8]:
for w in words[:1]:
    for ch1, ch2 in zip(w, w[1:]):
        print(ch1, ch2)

e m
m m
m a


## Load the dataset

The lecture uses a newline-delimited list of baby names. Keep a held-out split from the start so the experiment has a clean validation metric.


In [ ]:
words = load_words(DATA_PATH)
splits = split_words(words, seed=SEED)
train_words = splits["train"]
val_words = splits["val"]
test_words = splits["test"]
vocab = build_vocab(words)

stats = {
    "total_names": len(words),
    "train": len(train_words),
    "val": len(val_words),
    "test": len(test_words),
    "vocab_size": vocab.size,
}
stats


## Count-based bigram baseline

Start with the smallest correct model: a smoothed count table and row-normalized transition probabilities.


In [ ]:
count_matrix, count_probs = count_bigram_matrix(train_words, vocab, smoothing=1.0)
count_metrics = {
    "train_nll": average_negative_log_likelihood(train_words, count_probs, vocab),
    "val_nll": average_negative_log_likelihood(val_words, count_probs, vocab),
    "test_nll": average_negative_log_likelihood(test_words, count_probs, vocab),
}
count_samples = sample_from_prob_matrix(count_probs, vocab, num_samples=20, seed=SEED)
count_metrics, count_samples[:10]


In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(count_matrix, cmap="Blues")
plt.title("Add-one smoothed bigram counts")
plt.xticks(range(vocab.size), [vocab.itos[i] for i in range(vocab.size)])
plt.yticks(range(vocab.size), [vocab.itos[i] for i in range(vocab.size)])
plt.show()


## Neural bigram comparison

This model still predicts the next character from only the current character. The only change is that transition scores are learned with gradient descent instead of filled directly from counts.


In [ ]:
logits, neural_probs, history, neural_metrics = train_neural_bigram(
    train_words,
    val_words,
    vocab,
    num_steps=300,
    learning_rate=30.0,
    weight_decay=1e-2,
    seed=SEED,
)
neural_metrics["test_nll"] = average_negative_log_likelihood(test_words, neural_probs, vocab)
neural_samples = sample_from_prob_matrix(neural_probs, vocab, num_samples=20, seed=SEED)
neural_metrics, neural_samples[:10]


In [ ]:
history_df = pd.DataFrame(history)
history_df.plot(
    x="step",
    y=["train_loss", "val_nll"],
    figsize=(8, 4),
    title="Neural bigram training history",
)
plt.ylabel("loss / NLL")
plt.grid(alpha=0.3)
plt.show()


## Controlled experiment: data regime

Shrink the training set while holding the vocabulary, split logic, and evaluation metric fixed. This isolates how each bigram parameterization behaves as data becomes scarce.


In [ ]:
results = evaluate_data_regimes(
    train_words,
    val_words,
    vocab,
    fractions=(0.01, 0.05, 0.25, 1.0),
    smoothing=1.0,
    num_steps=250,
    learning_rate=30.0,
    weight_decay=1e-2,
    seed=SEED,
)
results_df = pd.DataFrame(results)
results_df


In [ ]:
ax = results_df.plot(
    x="train_examples",
    y=["count_val_nll", "neural_val_nll"],
    marker="o",
    logx=True,
    figsize=(8, 4),
    title="Validation NLL across training-set sizes",
)
ax.set_ylabel("validation NLL")
ax.grid(alpha=0.3)
plt.show()


In [ ]:
# # Building Makemore Bigrams: Lecture 2 Notes

# ## System

# Character-level bigram language model over names.

# Primary reference:

# - Andrej Karpathy, *Neural Networks: Zero to Hero*, lecture 2
# - `makemore_part1_bigrams.ipynb`
# - `karpathy/makemore`

# ## Baseline Build

# Recreate the lecture 2 count-based bigram model with:

# - a shared boundary token `.` for start and stop,
# - a 27 x 27 count matrix,
# - add-one smoothing,
# - row-normalized transition probabilities,
# - ancestral sampling for generated names,
# - average negative log-likelihood on held-out names.

# This is the smallest correct system for the lecture.

# ## Research Delta

# Compare the count-based bigram against a learned neural bigram model across
# different train-set sizes.

# Baseline:

# - add-one smoothed count bigram

# Delta:

# - neural bigram with a learned 27 x 27 logit table

# Controlled variable:

# - amount of training data: `1%`, `5%`, `25%`, `100%`

# Primary metric:

# - validation average negative log-likelihood

# Secondary checks:

# - sample quality,
# - sample diversity,
# - names copied exactly from the training set,
# - length distribution of generated samples.

# ## Why This Delta Works

# It keeps the architecture family intentionally small. Both models are still
# bigram models, so the experiment isolates the difference between:

# - explicit counts with smoothing,
# - learned logits optimized by gradient descent.

# That makes the result easier to explain than jumping immediately to a much
# larger MLP.

# ## Hypothesis

# - At very small data sizes, the count model may be competitive because its
#   inductive bias is strong and smoothing is simple.
# - As data increases, the neural bigram should match or slightly exceed the
#   count model on held-out NLL.
# - Neither model should capture long-range structure well; sample failures
#   should still be visibly local.

# ## Notebook Flow

# 1. Load `data/names.txt`.
# 2. Create train/val/test splits.
# 3. Build the smoothed count bigram.
# 4. Measure NLL and sample names.
# 5. Train the neural bigram on the same split.
# 6. Compare validation NLL across train-set sizes.
# 7. End with one short finding table.

# ## Result Table Template

# | train fraction | train examples | count val NLL | neural val NLL | takeaway |
# | --- | ---: | ---: | ---: | --- |
# | 1% |  |  |  |  |
# | 5% |  |  |  |  |
# | 25% |  |  |  |  |
# | 100% |  |  |  |  |


## Findings

Capture the result in one compact table:

| train fraction | count val NLL | neural val NLL | takeaway |
| --- | ---: | ---: | --- |
| 1% |  |  |  |
| 5% |  |  |  |
| 25% |  |  |  |
| 100% |  |  |  |
